# 01 - PPO

PPO uses one sampled solution per problem, a separate Qwen3 scalar critic, full token-level GAE, symmetric ratio clipping, and value clipping. TRL 1.8 keeps PPO under `trl.experimental.ppo`; the subclass below uses its model/state initialization but overrides the neural-reward training loop so the reward is exact GSM8K answer correctness.

In [ ]:
# Hyperparameters
sft_checkpoint = "./checkpoints/sft_base"
dataset_id = "openai/gsm8k"
dataset_config = "main"
dataset_revision = "740312add88f781978c0658806c59bc2815b9866"
seed = 17
train_problems = 256
eval_problems = 64
num_epochs = 1
batch_size = 16
forward_batch_size = 4
num_ppo_epochs = 4
max_prompt_tokens = 384
max_completion_tokens = 256
temperature = 0.8
top_p = 1.0
actor_learning_rate = 1e-6
critic_learning_rate = 2e-6
weight_decay = 0.0
policy_clip_epsilon = 0.2
value_clip_epsilon = 0.2
kl_coefficient = 0.05
gamma = 1.0
gae_lambda = 0.95
max_grad_norm = 1.0
eval_every_steps = 8
eval_batch_size = 16
wandb_project = "swe-rl-ablations"
wandb_run_name = "ppo"
output_path = "./checkpoints/ppo"

optimizer_steps = train_problems // batch_size
assert batch_size == 16 and forward_batch_size == 4
assert batch_size % forward_batch_size == 0
assert train_problems == 256 and eval_problems == 64
assert policy_clip_epsilon == 0.2 and gamma == 1.0 and gae_lambda == 0.95
print(f"PPO updates: {optimizer_steps}; PPO epochs per rollout: {num_ppo_epochs}")

In [ ]:
from pathlib import Path
import json
import os
import re
import time

import torch
import torch.nn as nn
import wandb
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, set_seed
from trl.experimental.ppo import PPOConfig, PPOTrainer
from trl.trainer.utils import selective_log_softmax

## Checkpoint and data

Resolve the SFT checkpoint, set reproducibility, and select the fixed GSM8K train and evaluation subsets.

In [ ]:
project_root = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
checkpoint_path = (project_root / sft_checkpoint).resolve()
if not (checkpoint_path / "config.json").exists():
    raise FileNotFoundError("Run 00_sft.ipynb first.")
os.environ["WANDB_PROJECT"] = wandb_project
set_seed(seed)
print(f"PPO base checkpoint: {checkpoint_path}")

train_rows = load_dataset(dataset_id, dataset_config, split="train", revision=dataset_revision).shuffle(seed=seed).select(range(train_problems))
eval_rows = load_dataset(dataset_id, dataset_config, split="test", revision=dataset_revision).shuffle(seed=seed).select(range(eval_problems))

## Exact-answer reward

Parse GSM8K answers and assign `1` for correct, `0` for incorrect, and `-1` for malformed completions.

In [ ]:
def gold_number(answer):
    match = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", answer)
    if not match:
        raise ValueError(f"Malformed GSM8K answer: {answer!r}")
    return match.group(1).replace(",", "")


def predicted_number(completion):
    match = re.search(r"Final answer:\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", completion.strip(), re.IGNORECASE)
    return None if not match else match.group(1).replace(",", "")


def exact_reward(completion, answer):
    prediction = predicted_number(completion)
    if prediction is None:
        return -1.0
    return 1.0 if float(prediction) == float(gold_number(answer)) else 0.0

## Prompt and tensor helpers

Format model prompts and provide the masked statistics used by PPO telemetry.

In [ ]:
def prompt_text(question):
    messages = [{"role": "user", "content": f"Solve this grade-school math problem. Show your reasoning, then end with `Final answer: <number>`.\n\n{question}"}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def masked_mean(values, mask):
    return (values * mask).sum() / mask.sum().clamp_min(1)


def gradient_norm(parameters):
    norms = [parameter.grad.detach().norm(2) for parameter in parameters if parameter.grad is not None]
    return float(torch.norm(torch.stack(norms), 2).item()) if norms else 0.0


def explained_variance(prediction, target, mask):
    prediction = prediction[mask.bool()]
    target = target[mask.bool()]
    variance = target.var(unbiased=False)
    return 0.0 if variance <= 1e-12 else float((1 - (target - prediction).var(unbiased=False) / variance).item())

## Metrics persistence

Write metrics atomically so an interrupted run never leaves a partially written JSON file.

In [ ]:
def write_metrics(path, document):
    temporary_path = path.with_suffix(".json.tmp")
    temporary_path.write_text(json.dumps(document, indent=2) + "\n")
    temporary_path.replace(path)

## Tokenizer, actor, reference, and critic

Load the trainable actor, frozen SFT reference, and separately trainable scalar-value critic.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, use_fast=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

actor = AutoModelForCausalLM.from_pretrained(checkpoint_path, dtype=torch.bfloat16, attn_implementation="flash_attention_2")
reference = AutoModelForCausalLM.from_pretrained(checkpoint_path, dtype=torch.bfloat16, attn_implementation="flash_attention_2")
critic = AutoModelForSequenceClassification.from_pretrained(
    checkpoint_path,
    num_labels=1,
    ignore_mismatched_sizes=True,
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
critic.config.pad_token_id = tokenizer.pad_token_id
for parameter in reference.parameters():
    parameter.requires_grad_(False)
reference.eval()

## TRL reward-model adapter

`PPOTrainer` requires a reward module during construction. This placeholder is never used because decoded GSM8K answers supply the actual rewards.

In [ ]:
class UnusedNeuralReward(nn.Module):
    """PPOTrainer requires a reward module, but exact rewards are computed from decoded answers."""
    def __init__(self):
        super().__init__()
        self.anchor = nn.Parameter(torch.zeros(()), requires_grad=False)

unused_reward_model = UnusedNeuralReward()

## Rollout generation

Generate one sampled completion per problem and construct masks that include completion tokens only through the first EOS token.

In [ ]:
def generate_rollout(model, rows):
    prompts = [prompt_text(question) for question in rows["question"]]
    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=max_prompt_tokens,
    ).to(model.device)
    generated = model.generate(
        **encoded,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_completion_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    completion = generated[:, max_prompt_tokens:]
    if completion.shape[1] < max_completion_tokens:
        padding = torch.full(
            (completion.shape[0], max_completion_tokens - completion.shape[1]),
            tokenizer.pad_token_id,
            device=completion.device,
        )
        completion = torch.cat([completion, padding], dim=1)
    eos = completion.eq(tokenizer.eos_token_id)
    first_eos = torch.where(
        eos.any(1),
        eos.float().argmax(1),
        torch.full((len(prompts),), max_completion_tokens - 1, device=completion.device),
    ).long()
    positions = torch.arange(max_completion_tokens, device=completion.device).unsqueeze(0)
    generated_mask = positions <= first_eos.unsqueeze(1)
    input_ids = torch.cat([encoded.input_ids, completion], dim=1)
    attention_mask = torch.cat([encoded.attention_mask, generated_mask.long()], dim=1)
    completion_mask = torch.zeros((len(prompts), input_ids.shape[1] - 1), device=model.device)
    completion_mask[:, max_prompt_tokens - 1:] = generated_mask.float()
    text = tokenizer.batch_decode(completion, skip_special_tokens=True)
    return input_ids, attention_mask, completion_mask, text

## Memory-bounded forward passes

Compute token log-probabilities and critic values in four-example chunks while preserving the logical rollout batch of 16.

In [ ]:
def token_logprobs(model, input_ids, attention_mask, requires_grad):
    chunks = []
    context = torch.enable_grad() if requires_grad else torch.no_grad()
    with context:
        for start in range(0, len(input_ids), forward_batch_size):
            ids = input_ids[start:start + forward_batch_size]
            mask = attention_mask[start:start + forward_batch_size]
            logits = model(input_ids=ids, attention_mask=mask, use_cache=False).logits[:, :-1]
            chunks.append(selective_log_softmax(logits, ids[:, 1:]))
    return torch.cat(chunks)


def token_values(model, input_ids, attention_mask, requires_grad):
    chunks = []
    context = torch.enable_grad() if requires_grad else torch.no_grad()
    with context:
        for start in range(0, len(input_ids), forward_batch_size):
            ids = input_ids[start:start + forward_batch_size]
            mask = attention_mask[start:start + forward_batch_size]
            output = model(input_ids=ids, attention_mask=mask, output_hidden_states=True, use_cache=False, return_dict=True)
            chunks.append(model.score(output.hidden_states[-1][:, :-1]).squeeze(-1).float())
    return torch.cat(chunks)

## Generalized advantage estimation

Propagate token rewards backward through each completion and normalize advantages over valid completion tokens.

In [ ]:
def full_gae(token_rewards, values, mask):
    advantages = torch.zeros_like(values)
    running = torch.zeros(len(values), device=values.device)
    for index in range(values.shape[1] - 1, -1, -1):
        next_value = values[:, index + 1] * mask[:, index + 1] if index + 1 < values.shape[1] else 0.0
        delta = token_rewards[:, index] + gamma * next_value - values[:, index]
        running = (delta + gamma * gae_lambda * running) * mask[:, index]
        advantages[:, index] = running
    returns = advantages + values
    selected = advantages[mask.bool()]
    advantages = (advantages - selected.mean()) / (selected.std(unbiased=False) + 1e-8)
    advantages = advantages * mask
    return advantages, returns

## Exact-answer PPO trainer

The trainer below keeps PPOTrainer's official initialization and public `.train()` entry point. The override is explicit because the stock implementation assumes a neural reward model and combines actor/critic optimization, which would make the requested exact reward and separate timings dishonest.

In [ ]:
class ExactAnswerPPOTrainer(PPOTrainer):
    def __init__(self, *args, exact_train_rows, exact_eval_rows, **kwargs):
        super().__init__(*args, **kwargs)
        self.exact_train_rows = exact_train_rows
        self.exact_eval_rows = exact_eval_rows

    @torch.no_grad()
    def exact_evaluate(self):
        policy = self.accelerator.unwrap_model(self.model).policy
        policy.eval()
        rewards = []
        for start in range(0, len(self.exact_eval_rows), eval_batch_size):
            stop = min(start + eval_batch_size, len(self.exact_eval_rows))
            rows = self.exact_eval_rows.select(range(start, stop))
            prompts = [prompt_text(question) for question in rows["question"]]
            encoded = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_prompt_tokens).to(policy.device)
            generated = policy.generate(**encoded, do_sample=False, max_new_tokens=max_completion_tokens, pad_token_id=tokenizer.pad_token_id, use_cache=True)
            text = tokenizer.batch_decode(generated[:, encoded.input_ids.shape[1]:], skip_special_tokens=True)
            rewards.extend(exact_reward(completion, answer) for completion, answer in zip(text, rows["answer"]))
        policy.train()
        return sum(rewards) / len(rewards)

    def train(self):
        wrapped = self.accelerator.unwrap_model(self.model)
        policy = wrapped.policy
        value_model = wrapped.value_model
        policy.train()
        value_model.train()
        policy_optimizer = torch.optim.AdamW(policy.parameters(), lr=actor_learning_rate, weight_decay=weight_decay)
        critic_optimizer = torch.optim.AdamW(value_model.parameters(), lr=critic_learning_rate, weight_decay=weight_decay)
        output_dir = (project_root / output_path).resolve()
        output_dir.mkdir(parents=True, exist_ok=True)
        metrics_path = output_dir / "metrics.json"
        metrics_document = {
            "base_checkpoint": str(checkpoint_path),
            "batch_size": batch_size,
            "forward_batch_size": forward_batch_size,
            "total_problems": len(self.exact_train_rows),
            "total_updates": optimizer_steps,
            "ppo_epochs_per_update": self.args.num_ppo_epochs,
            "steps": [],
        }
        run = wandb.init(project=wandb_project, name=wandb_run_name)
        torch.cuda.reset_peak_memory_stats()
        training_started = time.perf_counter()
        evaluation_started = time.perf_counter()
        initial_eval_reward = self.exact_evaluate()
        initial_metrics = {
            "step": 0,
            "eval/reward": initial_eval_reward,
            "time/evaluation_s": time.perf_counter() - evaluation_started,
        }
        metrics_document["steps"].append(initial_metrics)
        write_metrics(metrics_path, metrics_document)
        run.log({key: value for key, value in initial_metrics.items() if key != "step"}, step=0)
        last_eval_reward = initial_eval_reward

        global_step = 0
        for start in range(0, len(self.exact_train_rows), batch_size):
            step_started = time.perf_counter()
            rows = self.exact_train_rows.select(range(start, start + batch_size))
            torch.cuda.synchronize()
            rollout_started = time.perf_counter()
            with torch.no_grad():
                generation_started = time.perf_counter()
                input_ids, attention_mask, completion_mask, completions = generate_rollout(policy, rows)
                torch.cuda.synchronize()
                generation_seconds = time.perf_counter() - generation_started
                reward = torch.tensor(
                    [exact_reward(completion, answer) for completion, answer in zip(completions, rows["answer"])],
                    device=policy.device,
                )
                old_logps = token_logprobs(policy, input_ids, attention_mask, requires_grad=False)
                reference_logps = token_logprobs(self.ref_model, input_ids, attention_mask, requires_grad=False)
                old_values = token_values(value_model, input_ids, attention_mask, requires_grad=False)
                token_rewards = -self.args.kl_coef * (old_logps - reference_logps) * completion_mask
                final_positions = (torch.arange(completion_mask.shape[1], device=policy.device).unsqueeze(0) * completion_mask.long()).max(1).values
                token_rewards[torch.arange(len(reward), device=policy.device), final_positions] += reward
                advantages, returns = full_gae(token_rewards, old_values, completion_mask)
            torch.cuda.synchronize()
            rollout_seconds = time.perf_counter() - rollout_started

            policy_losses = []
            critic_losses = []
            clipped_tokens = []
            policy_grad_norms = []
            critic_grad_norms = []
            policy_seconds = 0.0
            critic_seconds = 0.0

            for _ in range(self.args.num_ppo_epochs):
                indices = torch.randperm(batch_size, device=policy.device)
                token_count = completion_mask[indices].sum()
                token_count_value = float(token_count.item())

                torch.cuda.synchronize()
                started_at = time.perf_counter()
                policy_optimizer.zero_grad(set_to_none=True)
                policy_loss_numerator = 0.0
                clipped_token_count = 0.0
                for microbatch_start in range(0, batch_size, forward_batch_size):
                    microbatch_indices = indices[microbatch_start:microbatch_start + forward_batch_size]
                    microbatch_mask = completion_mask[microbatch_indices]
                    new_logps = token_logprobs(policy, input_ids[microbatch_indices], attention_mask[microbatch_indices], requires_grad=True)
                    ratio = torch.exp(new_logps - old_logps[microbatch_indices])
                    unclipped = ratio * advantages[microbatch_indices]
                    clipped = ratio.clamp(1 - self.args.cliprange, 1 + self.args.cliprange) * advantages[microbatch_indices]
                    loss_numerator = -(torch.minimum(unclipped, clipped) * microbatch_mask).sum()
                    (loss_numerator / token_count).backward()
                    policy_loss_numerator += float(loss_numerator.detach().item())
                    controlling_clip = ((advantages[microbatch_indices] > 0) & (ratio > 1 + self.args.cliprange)) | ((advantages[microbatch_indices] < 0) & (ratio < 1 - self.args.cliprange))
                    clipped_token_count += float((controlling_clip.float() * microbatch_mask).sum().item())
                    del new_logps, ratio, unclipped, clipped, loss_numerator
                policy_grad_norms.append(gradient_norm(policy.parameters()))
                torch.nn.utils.clip_grad_norm_(policy.parameters(), self.args.max_grad_norm)
                policy_optimizer.step()
                torch.cuda.synchronize()
                policy_seconds += time.perf_counter() - started_at
                clipped_tokens.append(clipped_token_count / token_count_value)
                policy_losses.append(policy_loss_numerator / token_count_value)
                torch.cuda.empty_cache()

                torch.cuda.synchronize()
                started_at = time.perf_counter()
                critic_optimizer.zero_grad(set_to_none=True)
                value_loss_numerator = 0.0
                for microbatch_start in range(0, batch_size, forward_batch_size):
                    microbatch_indices = indices[microbatch_start:microbatch_start + forward_batch_size]
                    microbatch_mask = completion_mask[microbatch_indices]
                    new_values = token_values(value_model, input_ids[microbatch_indices], attention_mask[microbatch_indices], requires_grad=True)
                    clipped_values = old_values[microbatch_indices] + (new_values - old_values[microbatch_indices]).clamp(-self.args.cliprange_value, self.args.cliprange_value)
                    loss_numerator = 0.5 * (
                        torch.maximum((new_values - returns[microbatch_indices]).square(), (clipped_values - returns[microbatch_indices]).square()) * microbatch_mask
                    ).sum()
                    (loss_numerator / token_count).backward()
                    value_loss_numerator += float(loss_numerator.detach().item())
                    del new_values, clipped_values, loss_numerator
                critic_grad_norms.append(gradient_norm(value_model.parameters()))
                torch.nn.utils.clip_grad_norm_(value_model.parameters(), self.args.max_grad_norm)
                critic_optimizer.step()
                torch.cuda.synchronize()
                critic_seconds += time.perf_counter() - started_at
                critic_losses.append(value_loss_numerator / token_count_value)
                torch.cuda.empty_cache()

            global_step += 1
            self.state.global_step = global_step
            self.state.episode = global_step * batch_size
            self.state.epoch = self.state.episode / len(self.exact_train_rows)
            final_values = token_values(value_model, input_ids, attention_mask, requires_grad=False)
            post_update_logps = token_logprobs(policy, input_ids, attention_mask, requires_grad=False)
            evaluation_seconds = 0.0
            eval_reward = None
            if global_step % eval_every_steps == 0 or global_step == optimizer_steps:
                evaluation_started = time.perf_counter()
                eval_reward = self.exact_evaluate()
                evaluation_seconds = time.perf_counter() - evaluation_started
            step_seconds = time.perf_counter() - step_started
            elapsed_seconds = time.perf_counter() - training_started
            eta_seconds = elapsed_seconds / global_step * (optimizer_steps - global_step)
            metrics = {
                "progress/problems_completed": global_step * batch_size,
                "progress/updates_completed": global_step,
                "train/reward": float(reward.mean().item()),
                "loss/policy": sum(policy_losses) / len(policy_losses),
                "objective/kl_sft": float(masked_mean(post_update_logps - reference_logps, completion_mask).item()),
                "policy/tokens_clipped_or_masked_pct": 100 * sum(clipped_tokens) / len(clipped_tokens),
                "critic/loss": sum(critic_losses) / len(critic_losses),
                "critic/explained_variance": explained_variance(final_values, returns, completion_mask),
                "grad_norm/policy": sum(policy_grad_norms) / len(policy_grad_norms),
                "grad_norm/critic": sum(critic_grad_norms) / len(critic_grad_norms),
                "time/policy_forward_backward_s": policy_seconds,
                "time/critic_forward_backward_s": critic_seconds,
                "time/generation_s": generation_seconds,
                "time/generation_per_problem_s": generation_seconds / len(rows),
                "time/rollout_s": rollout_seconds,
                "time/rollout_per_problem_s": rollout_seconds / len(rows),
                "time/evaluation_s": evaluation_seconds,
                "time/step_s": step_seconds,
                "time/step_per_problem_s": step_seconds / len(rows),
                "time/elapsed_s": elapsed_seconds,
                "time/eta_s": eta_seconds,
                "gpu/peak_memory_gb": torch.cuda.max_memory_allocated() / 1024**3,
            }
            if eval_reward is not None:
                last_eval_reward = eval_reward
                metrics["eval/reward"] = eval_reward
            metrics_document["steps"].append({"step": global_step, **metrics})
            metrics_document["summary"] = {
                "updates_completed": global_step,
                "problems_completed": global_step * batch_size,
                "elapsed_s": elapsed_seconds,
                "eta_s": eta_seconds,
                "latest_train_reward": metrics["train/reward"],
                "latest_eval_reward": last_eval_reward,
            }
            write_metrics(metrics_path, metrics_document)
            run.log(metrics, step=global_step)

        metrics_document["summary"]["completed"] = True
        metrics_document["summary"]["total_time_s"] = time.perf_counter() - training_started
        write_metrics(metrics_path, metrics_document)
        policy.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
        run.finish()
        return self.state

## TRL dataset adapter

Tokenize prompts for the dataset interface expected by `PPOTrainer`; the custom loop still uses the original GSM8K rows.

In [ ]:
tokenized_prompts = train_rows.map(
    lambda row: tokenizer(prompt_text(row["question"]), truncation=True, max_length=max_prompt_tokens),
    remove_columns=train_rows.column_names,
)

## PPO configuration

Create the official TRL configuration object. The custom trainer reads its clipping, KL, GAE, epoch, and gradient settings.

In [ ]:
ppo_args = PPOConfig(
    output_dir=str(project_root / output_path),
    run_name=wandb_run_name,
    sft_model_path=str(checkpoint_path),
    num_train_epochs=num_epochs,
    total_episodes=train_problems,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=1,
    local_rollout_forward_batch_size=batch_size,
    response_length=max_completion_tokens,
    stop_token="eos",
    temperature=temperature,
    learning_rate=actor_learning_rate,
    num_ppo_epochs=num_ppo_epochs,
    num_mini_batches=1,
    num_sample_generations=0,
    cliprange=policy_clip_epsilon,
    cliprange_value=value_clip_epsilon,
    kl_coef=kl_coefficient,
    gamma=gamma,
    lam=gae_lambda,
    max_grad_norm=max_grad_norm,
    seed=seed,
    report_to=[],
    bf16=True,
)

## Trainer construction

Connect the actor, frozen reference, critic, tokenizer, and exact GSM8K datasets.

In [ ]:
trainer = ExactAnswerPPOTrainer(
    args=ppo_args,
    processing_class=tokenizer,
    model=actor,
    ref_model=reference,
    reward_model=unused_reward_model,
    train_dataset=tokenized_prompts,
    value_model=critic,
    exact_train_rows=train_rows,
    exact_eval_rows=eval_rows,
)

## Run training

Execute PPO. If training fails, persist the error in `metrics.json`, close W&B cleanly, release cached GPU memory, and re-raise the exception.

In [ ]:
try:
    trainer.train()
except Exception as error:
    failure_output_dir = (project_root / output_path).resolve()
    failure_output_dir.mkdir(parents=True, exist_ok=True)
    failure_metrics_path = failure_output_dir / "metrics.json"
    if failure_metrics_path.exists():
        failure_document = json.loads(failure_metrics_path.read_text())
    else:
        failure_document = {"base_checkpoint": str(checkpoint_path), "steps": []}
    failure_document["summary"] = {
        **failure_document.get("summary", {}),
        "completed": False,
        "error_type": type(error).__name__,
        "error_message": str(error),
    }
    write_metrics(failure_metrics_path, failure_document)
    if wandb.run is not None:
        wandb.finish(exit_code=1)
    torch.cuda.empty_cache()
    raise